## Example: Iterative Bayesian updating with a Gaussian mean model (RWMH posterior + PPC)

### Goal
We simulate 3 incoming batches of data. After each batch we:
1) produce an updated posterior object over the parameter vector $\mu \in \mathbb{R}^2$,  
2) generate a small posterior predictive forecast, and  
3) compute a posterior predictive check (PPC) p-value based on a chosen statistic.

This is meant to showcase the “distributions-in → distributions-out” workflow pattern.

In [1]:
import numpy as np

from probpipe import (
    SimpleLikelihood,
    RWMH, 
    IterativeForecaster,
    PosteriorPredictiveChecker,
    Gaussian
    )


### Probabilistic model

**Unknown parameter**  
We want to infer the 2D mean vector $\mu \in \mathbb{R}^2$ from observed data. 
This represents the location parameter of a Gaussian distribution.


**Prior**  
$\mu \sim \mathcal{N}(m_0, \Sigma_0)$, where 
$m_0 = \begin{bmatrix}0\\0\end{bmatrix}$, 
$\Sigma_0 = I_2.$

This represents our initial belief: we expect $\mu$ to be near zero with unit variance in each dimension.

`prior` encodes $p(\mu) = \mathcal{N}(m_0,\Sigma_0)$.


In [2]:
prior = Gaussian(mean=np.array([0.0, 0.0]), cov=np.eye(2))


**Likelihood wrapper**

`SimpleLikelihood` defines a likelihood family where:

$x \mid \mu \sim \mathcal{N}(\mu, I_2)$,
i.e. the unknown parameter is plugged into the distribution’s `mean` field, while `cov` is fixed.

For a batch $D=\{x_1,\dots,x_n\}$,
$\log p(D\mid\mu)=\sum_{j=1}^n \log \mathcal{N}(x_j\mid \mu, I_2)$.

In [3]:
likelihood = SimpleLikelihood(dist_cls=Gaussian, params_name="mean", cov=np.eye(2))

**Posterior**  
$p(\mu \mid D) \propto p(D \mid \mu)\,p(\mu)$.

**Note:** This model is conjugate (Gaussian prior + Gaussian likelihood), so we could compute the posterior analytically. However, we intentionally use MCMC to demonstrate the ProbPipe workflow for more general cases where closed-form solutions don't exist.


### `RWMH` - Random Walk Metropolis-Hastings

The `RWMH` class implements a Random Walk Metropolis-Hastings sampler that:
1. Runs MCMC with proposal $\mu' = \mu + \epsilon$, where $\epsilon \sim \mathcal{N}(0, \sigma^2 I_2)$ (step_size = $\sigma$)
2. Accepts/rejects proposals using the Metropolis-Hastings criterion
3. Generates ~1,600 posterior samples (after burn-in and thinning)
4. Fits a Gaussian approximation $q(\mu\mid D) \approx \mathcal{N}(\hat\mu, \hat\Sigma)$ to the MCMC samples

Where:
- $\hat\mu$ = empirical mean of MCMC samples
- $\hat\Sigma$ = empirical covariance of MCMC samples

This Gaussian approximation enables efficient iterative updating since it provides a closed-form `log_density()` for use as a prior in subsequent iterations.


In [4]:
approx_post = RWMH(step_size=0.1)

### Iterative Forecaster

`IterativeForecaster` orchestrates the sequential Bayesian updating workflow:

**Update step (`.update(data=obs_data)`):**
- Takes new data and current posterior (from previous iteration)
- Uses the current posterior as the prior for the new iteration
- Runs RWMH to compute updated posterior: $p(\mu \mid D_1, \ldots, D_t)$
- Returns a `PosteriorDistribution` wrapping the Gaussian approximation

**Forecast step (`.forecast(n_samples)`):**
- Samples one parameter value: $\mu^{(1)} \sim q(\mu \mid D_{1:t})$
- Generates synthetic observations: $x^{*}_1,\ldots,x^{*}_{n} \sim \mathcal{N}(\mu^{(1)}, I_2)$
- Returns the forecasted observations (posterior predictive samples)

**Iterative property:** Each iteration's posterior becomes the next iteration's prior, enabling proper sequential Bayesian learning:
$$p(\mu \mid D_1, D_2) \propto p(D_2 \mid \mu) \cdot p(\mu \mid D_1)$$


In [5]:
forecaster = IterativeForecaster(prior=prior, likelihood=likelihood, approx_post=approx_post)

### Posterior Predictive Check (PPC)

The PPC assesses model fit by comparing observed data to synthetic data from the posterior predictive distribution.

Given a test statistic $T(\cdot)$ (e.g., mean), the checker computes:

1. **Observed statistic:** $T_{\text{obs}} = T(D_{\text{obs}})$ from the actual data

2. **Simulated statistics:** For $k=1,\ldots,200$ replicates:
   - Draw parameter: $\mu^{(k)} \sim q(\mu \mid D_{\text{obs}})$
   - Generate synthetic data: $D^{\text{rep}}_k \sim p(\cdot \mid \mu^{(k)})$ 
   - Compute: $T_k = T(D^{\text{rep}}_k)$

3. **P-value:** $p_{\text{PPC}} = \frac{1}{200}\sum_{k=1}^{200}\mathbf{1}\{T_k \ge T_{\text{obs}}\}$

**Interpretation:** 
- $p_{\text{PPC}} \approx 0.5$ suggests the model fits well (observed statistic is typical)
- Extreme values (near 0 or 1) suggest model misspecification

**Note:** This implementation uses `n_samples=1` for each replicate, while the observed statistic is computed on the full batch (100 points). For a more rigorous PPC, match the sample sizes.


In [6]:
ppc = PosteriorPredictiveChecker(statistic=np.mean)

### Data generation in the loop

Each iteration simulates a new batch of size $n=100$ from:
$x_j \sim \mathcal{N}\left(\begin{bmatrix}5\\-3\end{bmatrix},\,4I_2\right)$.

Meanwhile the likelihood inside the model uses covariance $I_2$, so the model is intentionally mis-specified in noise scale (it assumes less variance than the data generator).

In [7]:
forecasts = []
ppc_pvalues = []

for i in range(3):
    obs_data = np.random.multivariate_normal(mean=np.array([5.0, -3.0]), cov=4*np.eye(2), size=100)
    posterior_dist = forecaster.update(data=obs_data)
    forecast = forecaster.forecast(n_samples=10)

    forecasts.append(forecast)
    ppc_pvalues.append(ppc.predictive_p_value(posterior=posterior_dist))

    print(f"Posterior distribution after dataset {i}: {posterior_dist}")
    print(f"[{i}] ppc p-value: {ppc_pvalues[-1]}")
    

Posterior distribution after dataset 0: Gaussian(mu = [ 4.71082403 -3.03295177], cov = [[ 1.05081972e-02 -7.69922627e-06]
 [-7.69922627e-06  1.01211423e-02]])
[0] ppc p-value: 0.47
Posterior distribution after dataset 1: Gaussian(mu = [ 4.89500098 -3.03433443], cov = [[0.00506065 0.00013928]
 [0.00013928 0.00534324]])
[1] ppc p-value: 0.43
Posterior distribution after dataset 2: Gaussian(mu = [ 4.8054547  -3.03477395], cov = [[ 3.21092061e-03 -9.81422769e-05]
 [-9.81422769e-05  3.46462759e-03]])
[2] ppc p-value: 0.54
